# Day 6：从零手写 GRPO 训练（面试 Tier 2！）

> **目标**：从零实现完整的 GRPO 训练循环——`Reward 函数设计 → 多组采样 → 组内归一化 Advantage → PPO-Clip Loss + KL Penalty → 训练循环 → 效果可视化`。在简单的数学/格式任务上训练 GPT-2，观察策略在 GRPO 训练过程中的改善。
>
> GRPO 的核心在于「用组内对比替代 Critic」，这使得代码比 RLHF-PPO 简洁得多（无需 Critic 网络、无需 GAE），同时保持了 on-policy RL 的探索能力。

---

## 实现路线图

```
Part 1: 环境准备与数学回顾
  导入库 → 设备设置 → GRPO Loss 公式回顾

Part 2: Reward 函数设计
  规则验证 reward → 格式奖励 + 内容奖励

Part 3: 多组采样
  对同一 prompt 生成 G 个 response → 计算 reward

Part 4: 组内归一化 Advantage
  A_i = (r_i - mean(r)) / std(r)

Part 5: GRPO Loss 实现
  PPO-Clip + KL Penalty → 完整 loss 函数

Part 6: 完整训练循环
  采样 → 计算 Advantage → GRPO 更新 → 监控指标

Part 7: 训练效果可视化
  reward 曲线 → 生成质量对比 → Advantage 分布
```

In [ ]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

try:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers'])
    from transformers import GPT2LMHeadModel, GPT2Tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Part 1：GRPO Loss 数学回顾

Day 5 推导的 GRPO Loss：

$$L_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G}\left[\min\left(\rho_i \hat{A}_i, \text{clip}(\rho_i, 1\pm\epsilon) \hat{A}_i\right) - \beta \hat{D}_{\text{KL},i}\right]$$

其中：
- $\rho_i = \frac{\pi_\theta(y_i|x)}{\pi_{\theta_{\text{old}}}(y_i|x)}$ 是 importance sampling ratio
- $\hat{A}_i = \frac{r_i - \bar{r}}{\sigma_r}$ 是组内归一化 Advantage
- $\hat{D}_{\text{KL},i}$ 是策略与参考模型之间的 KL 散度

## Part 2：Reward 函数设计

我们设计一个简单的规则 reward 来验证 GRPO 的训练效果。

任务：给定 prompt，模型需要生成符合特定质量标准的 response。

Reward 规则：
- **长度奖励**：response 长度在合理范围内（20-80 tokens）
- **多样性奖励**：response 中不同 token 的比例
- **格式奖励**：response 不以重复开头、不含过多重复 n-gram

In [ ]:
def compute_reward(response_text, response_ids):
    """
    规则验证的 reward 函数。
    模拟 DeepSeek-R1 中基于规则的 reward 设计（Day 4）。
    
    Returns:
        reward: float, 总 reward
    """
    reward = 0.0
    
    # 长度奖励: 鼓励 20-80 token 的 response
    length = len(response_ids)
    if 20 <= length <= 80:
        reward += 1.0
    elif 10 <= length < 20 or 80 < length <= 120:
        reward += 0.3
    else:
        reward -= 0.5
    
    # 多样性奖励: unique token 比例
    if length > 0:
        unique_ratio = len(set(response_ids)) / length
        reward += unique_ratio * 1.0  # max +1.0
    
    # 重复惩罚: 检查 3-gram 重复
    if length >= 3:
        trigrams = [tuple(response_ids[i:i+3]) for i in range(length - 2)]
        unique_trigrams = len(set(trigrams))
        total_trigrams = len(trigrams)
        if total_trigrams > 0:
            repeat_ratio = 1.0 - unique_trigrams / total_trigrams
            if repeat_ratio > 0.5:
                reward -= 1.0
    
    # 非空奖励
    if length > 5 and len(response_text.strip()) > 10:
        reward += 0.5
    
    return reward

# 测试 reward 函数
test_cases = [
    ("This is a good detailed answer about machine learning concepts.", list(range(30))),
    ("OK", list(range(3))),
    ("aaa " * 50, [1] * 50),
]
for text, ids in test_cases:
    r = compute_reward(text, ids)
    print(f"Reward: {r:+.2f} | Length: {len(ids):3d} | Text: {text[:50]}...")

In [ ]:
# Prompt 集合
PROMPTS = [
    "Explain what machine learning is.",
    "Describe how neural networks work.",
    "What is deep learning?",
    "How does gradient descent optimize a model?",
    "What is natural language processing?",
    "Explain reinforcement learning.",
    "What is a transformer model?",
    "Describe the concept of overfitting.",
    "How does backpropagation work?",
    "What is transfer learning?",
    "Explain the attention mechanism.",
    "What is a loss function?",
]

print(f"Number of prompts: {len(PROMPTS)}")

## Part 3：模型初始化与采样工具

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

policy_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ref_model = copy.deepcopy(policy_model).to(device)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in policy_model.parameters())
print(f"Policy Model: {total_params/1e6:.1f}M params")
print(f"Reference Model: {total_params/1e6:.1f}M params (frozen)")
print(f"No Critic needed! (GRPO advantage)")

In [ ]:
def generate_responses(model, tokenizer, prompt, num_samples, max_new_tokens=60):
    """
    对同一 prompt 生成 G 个 response（GRPO 的核心采样步骤）。
    
    Returns:
        responses: list of (response_text, response_ids, full_ids, prompt_len)
    """
    model.eval()
    prompt_text = f"Question: {prompt}\nAnswer:"
    prompt_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    prompt_len = prompt_ids.shape[1]
    
    responses = []
    for _ in range(num_samples):
        with torch.no_grad():
            output = model.generate(
                prompt_ids,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        full_ids = output[0].tolist()
        response_ids = full_ids[prompt_len:]
        response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
        
        responses.append({
            "text": response_text,
            "response_ids": response_ids,
            "full_ids": full_ids,
            "prompt_len": prompt_len,
        })
    
    return responses

# 测试采样
test_responses = generate_responses(policy_model, tokenizer, PROMPTS[0], num_samples=4)
print(f"Prompt: {PROMPTS[0]}")
for i, resp in enumerate(test_responses):
    r = compute_reward(resp["text"], resp["response_ids"])
    print(f"  Response {i}: (reward={r:+.2f}, len={len(resp['response_ids'])}) {resp['text'][:80]}...")

## Part 4：组内归一化 Advantage

对应 Day 5 公式：

$$\hat{A}_i = \frac{r_i - \bar{r}}{\sigma_r + \epsilon}$$

In [ ]:
def compute_group_advantages(rewards, eps=1e-8):
    """
    计算组内归一化 Advantage。
    
    对应 Day 5 公式:
    A_i = (r_i - mean(r)) / (std(r) + eps)
    
    Args:
        rewards: tensor [G], 组内每个 response 的 reward
    Returns:
        advantages: tensor [G], 归一化 Advantage
    """
    mean_r = rewards.mean()
    std_r = rewards.std()
    advantages = (rewards - mean_r) / (std_r + eps)
    return advantages

# 测试
test_rewards = torch.tensor([2.0, 1.5, 0.5, -0.5])
test_adv = compute_group_advantages(test_rewards)
print(f"Rewards:    {test_rewards.tolist()}")
print(f"Advantages: {[f'{a:.3f}' for a in test_adv.tolist()]}")
print(f"Mean of advantages: {test_adv.mean():.6f} (should be ~0)")

## Part 5：GRPO Loss 实现

对应 Day 5 推导的 GRPO Loss：

$$L_{\text{GRPO}} = -\frac{1}{G}\sum_{i=1}^{G}\left[\min(\rho_i A_i, \text{clip}(\rho_i, 1\pm\epsilon) A_i) - \beta \hat{D}_{\text{KL},i}\right]$$

In [ ]:
def get_sequence_log_probs(model, full_ids_list, prompt_len, max_len=128):
    """
    计算一组 response 的 sequence-level log probability。
    对 response 部分的 per-token log prob 求和。
    """
    # Pad sequences to same length
    padded = []
    masks = []
    resp_masks = []
    for ids in full_ids_list:
        ids = ids[:max_len]
        seq_len = len(ids)
        pad_len = max_len - seq_len
        padded.append(ids + [tokenizer.pad_token_id] * pad_len)
        masks.append([1] * seq_len + [0] * pad_len)
        resp_mask = [0] * prompt_len + [1] * (seq_len - prompt_len) + [0] * pad_len
        resp_masks.append(resp_mask)
    
    input_ids = torch.tensor(padded, dtype=torch.long, device=device)
    attention_mask = torch.tensor(masks, dtype=torch.long, device=device)
    response_mask = torch.tensor(resp_masks, dtype=torch.float, device=device)
    
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
    target_ids = input_ids[:, 1:]
    per_token_logps = log_probs.gather(dim=-1, index=target_ids.unsqueeze(-1)).squeeze(-1)
    
    resp_mask_shifted = response_mask[:, 1:]
    seq_logps = (per_token_logps * resp_mask_shifted).sum(dim=-1)
    
    return seq_logps, per_token_logps, resp_mask_shifted

In [ ]:
def grpo_loss(policy_model, ref_model, responses, rewards, prompt_len,
              old_log_probs, beta=0.1, clip_eps=0.2):
    """
    计算 GRPO Loss。
    
    对应 Day 5 公式:
    L = -1/G * Σ [min(ρ_i * A_i, clip(ρ_i) * A_i) - β * KL_i]
    
    Args:
        policy_model: 当前策略 π_θ
        ref_model: 参考策略 π_ref (冻结)
        responses: list of response dicts
        rewards: tensor [G] of rewards
        prompt_len: int, prompt 长度
        old_log_probs: tensor [G] of old policy log probs
        beta: KL penalty coefficient
        clip_eps: PPO clip epsilon
    
    Returns:
        loss: scalar
        metrics: dict
    """
    G = len(responses)
    full_ids_list = [r["full_ids"] for r in responses]
    
    # ====== Step 1: 组内归一化 Advantage ======
    advantages = compute_group_advantages(rewards)
    
    # ====== Step 2: 当前策略的 log probs ======
    new_log_probs, _, _ = get_sequence_log_probs(policy_model, full_ids_list, prompt_len)
    
    # ====== Step 3: 参考模型的 log probs ======
    with torch.no_grad():
        ref_log_probs, _, _ = get_sequence_log_probs(ref_model, full_ids_list, prompt_len)
    
    # ====== Step 4: Importance sampling ratio ======
    log_ratio = new_log_probs - old_log_probs
    ratio = torch.exp(log_ratio)
    
    # ====== Step 5: PPO-Clip objective ======
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
    policy_loss = -torch.min(surr1, surr2).mean()
    
    # ====== Step 6: KL Penalty ======
    kl = (new_log_probs - ref_log_probs).mean()
    
    # ====== Step 7: Total Loss ======
    loss = policy_loss + beta * kl
    
    with torch.no_grad():
        metrics = {
            "loss": loss.item(),
            "policy_loss": policy_loss.item(),
            "kl": kl.item(),
            "mean_reward": rewards.mean().item(),
            "mean_advantage": advantages.mean().item(),
            "mean_ratio": ratio.mean().item(),
            "clip_fraction": ((ratio - 1.0).abs() > clip_eps).float().mean().item(),
        }
    
    return loss, metrics

## Part 6：完整 GRPO 训练循环

GRPO 训练循环的核心流程：

```
for each iteration:
    1. 采样：对每个 prompt 生成 G 个 response (π_old)
    2. 打分：计算每个 response 的 reward
    3. Advantage：组内归一化
    4. 更新：GRPO Loss → backward → optimizer.step()
    5. 同步：π_old ← π_θ
```

In [ ]:
# ====== 超参数 ======
G = 4                 # 组大小 (每个 prompt 采样 G 个 response)
NUM_ITERATIONS = 15   # 训练轮数
K_EPOCHS = 2          # 每轮 GRPO 更新次数
BETA = 0.05           # KL penalty 系数
CLIP_EPS = 0.2        # PPO clip epsilon
LR = 5e-6             # 学习率
PROMPTS_PER_ITER = 4  # 每轮使用的 prompt 数
MAX_NEW_TOKENS = 60   # 最大生成 token 数

optimizer = optim.AdamW(policy_model.parameters(), lr=LR, weight_decay=0.01)

history = {
    "mean_reward": [], "loss": [], "kl": [],
    "policy_loss": [], "clip_fraction": [],
}

print(f"=== GRPO Training ===")
print(f"Group size G={G}, Iterations={NUM_ITERATIONS}, K={K_EPOCHS}")
print(f"β={BETA}, ε={CLIP_EPS}, lr={LR}")
print(f"Prompts per iteration: {PROMPTS_PER_ITER}")
print(f"{'='*60}")

policy_old = copy.deepcopy(policy_model)

for iteration in range(NUM_ITERATIONS):
    iter_metrics = {k: [] for k in history}
    
    # ====== Phase 1: 采样 ======
    selected_prompts = random.sample(PROMPTS, min(PROMPTS_PER_ITER, len(PROMPTS)))
    
    all_data = []
    for prompt in selected_prompts:
        # 用旧策略采样 G 个 response
        policy_old.eval()
        responses = generate_responses(policy_old, tokenizer, prompt, G, MAX_NEW_TOKENS)
        
        # 计算 reward
        rewards = torch.tensor(
            [compute_reward(r["text"], r["response_ids"]) for r in responses],
            dtype=torch.float, device=device
        )
        
        # 计算旧策略的 log probs
        with torch.no_grad():
            old_logps, _, _ = get_sequence_log_probs(
                policy_old,
                [r["full_ids"] for r in responses],
                responses[0]["prompt_len"]
            )
        
        all_data.append({
            "responses": responses,
            "rewards": rewards,
            "old_log_probs": old_logps,
            "prompt_len": responses[0]["prompt_len"],
        })
    
    # ====== Phase 2: GRPO 更新 ======
    policy_model.train()
    for k_epoch in range(K_EPOCHS):
        for data in all_data:
            loss, metrics = grpo_loss(
                policy_model, ref_model,
                data["responses"], data["rewards"],
                data["prompt_len"], data["old_log_probs"],
                beta=BETA, clip_eps=CLIP_EPS
            )
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
            optimizer.step()
            
            for k in iter_metrics:
                iter_metrics[k].append(metrics[k])
    
    # ====== Phase 3: 同步旧策略 ======
    policy_old = copy.deepcopy(policy_model)
    
    # 记录
    for k in history:
        history[k].append(np.mean(iter_metrics[k]))
    
    if (iteration + 1) % 3 == 0 or iteration == 0:
        print(f"Iter {iteration+1:3d}/{NUM_ITERATIONS} | "
              f"Reward: {history['mean_reward'][-1]:.3f} | "
              f"Loss: {history['loss'][-1]:.4f} | "
              f"KL: {history['kl'][-1]:.4f} | "
              f"Clip%: {history['clip_fraction'][-1]:.2%}")

print(f"{'='*60}")
print(f"Training complete!")
print(f"Final Mean Reward: {history['mean_reward'][-1]:.3f}")

## Part 7：训练效果可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(history["mean_reward"])
axes[0, 0].set_title("Mean Reward")
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history["loss"])
axes[0, 1].set_title("GRPO Loss")
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history["kl"])
axes[1, 0].set_title("KL Divergence")
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history["clip_fraction"])
axes[1, 1].set_title("Clip Fraction")
axes[1, 1].set_xlabel("Iteration")
axes[1, 1].axhline(y=0.1, color='r', linestyle='--', alpha=0.5, label='target ~10%')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("GRPO Training Curves", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 训练后生成对比
print("=== Generation Comparison: Before vs After GRPO ===")
print("=" * 70)

for prompt in PROMPTS[:4]:
    ref_resp = generate_responses(ref_model, tokenizer, prompt, 1, MAX_NEW_TOKENS)[0]
    grpo_resp = generate_responses(policy_model, tokenizer, prompt, 1, MAX_NEW_TOKENS)[0]
    
    ref_r = compute_reward(ref_resp["text"], ref_resp["response_ids"])
    grpo_r = compute_reward(grpo_resp["text"], grpo_resp["response_ids"])
    
    print(f"\nPrompt: {prompt}")
    print(f"  [Reference] (r={ref_r:+.2f}) {ref_resp['text'][:100]}")
    print(f"  [GRPO]      (r={grpo_r:+.2f}) {grpo_resp['text'][:100]}")
    print("-" * 70)

In [ ]:
# 总结
print("=" * 60)
print("Day 6 Summary: GRPO Training from Scratch")
print("=" * 60)
print(f"\n1. Algorithm: GRPO (Group Relative Policy Optimization)")
print(f"2. Group size G: {G}")
print(f"3. Key advantage: No Critic network needed!")
print(f"4. Models: Policy + Reference only (2 models, vs 4 for PPO)")
print(f"5. Advantage estimation: Group-normalized rewards")
print(f"6. Training: {NUM_ITERATIONS} iterations × {K_EPOCHS} epochs")
print(f"7. Final Mean Reward: {history['mean_reward'][-1]:.3f}")
print(f"\nKey differences from DPO (Day 3):")
print(f"  - GRPO is on-policy (generates new responses each iteration)")
print(f"  - GRPO uses explicit reward function (not preference pairs)")
print(f"  - GRPO uses PPO-Clip (not log-sigmoid)")
print(f"  - GRPO better suited for verifiable tasks (math, code)")